# Лабораторна робота №2
**Тема:** Наука про дані: підготовчий етап

## Завдання 1: Завантаження даних
Завантажити тестові структуровані файли, що містять значення VHI-індексу для кожної області. При зберіганні файлу до його імені додати дату та час завантаження. Запобігти повторному завантаженню.

In [ ]:
import os
import urllib.request
import datetime
import glob

def download_noaa_data(target_dir="vhi_data"):

    os.makedirs(target_dir, exist_ok=True)
    
    now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for prov_id in range(1, 28):
        existing_files = glob.glob(f"{target_dir}/vhi_prov_{prov_id}_*.csv")
        if existing_files:
            print(f"Дані для області {prov_id} вже завантажено. Пропускаємо.")
            continue
            
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={prov_id}&year1=1981&year2=2024&type=Mean"
        filename = f"vhi_prov_{prov_id}_{now}.csv"
        filepath = os.path.join(target_dir, filename)
        
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req) as response:
                text = response.read().decode('utf-8')
                
            with open(filepath, 'w') as f:
                f.write(text)
            print(f"Успішно завантажено: {filename}")
            
        except Exception as e:
            print(f"Помилка завантаження для області {prov_id}: {e}")

download_noaa_data()

Успішно завантажено: vhi_prov_1_20260404_234910.csv
Успішно завантажено: vhi_prov_2_20260404_234910.csv
Успішно завантажено: vhi_prov_3_20260404_234910.csv
Успішно завантажено: vhi_prov_4_20260404_234910.csv
Успішно завантажено: vhi_prov_5_20260404_234910.csv
Успішно завантажено: vhi_prov_6_20260404_234910.csv
Успішно завантажено: vhi_prov_7_20260404_234910.csv
Успішно завантажено: vhi_prov_8_20260404_234910.csv
Успішно завантажено: vhi_prov_9_20260404_234910.csv
Успішно завантажено: vhi_prov_10_20260404_234910.csv
Успішно завантажено: vhi_prov_11_20260404_234910.csv
Успішно завантажено: vhi_prov_12_20260404_234910.csv
Успішно завантажено: vhi_prov_13_20260404_234910.csv
Успішно завантажено: vhi_prov_14_20260404_234910.csv
Успішно завантажено: vhi_prov_15_20260404_234910.csv
Успішно завантажено: vhi_prov_16_20260404_234910.csv
Успішно завантажено: vhi_prov_17_20260404_234910.csv
Успішно завантажено: vhi_prov_18_20260404_234910.csv
Успішно завантажено: vhi_prov_19_20260404_234910.csv
Ус

## Завдання 2: Зчитування та очищення даних
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, видалити зайвий текст. Замінити індекси так, щоб області індексувалася за українською абеткою.

In [5]:
import pandas as pd

def load_and_clean_data(data_dir="vhi_data"):
    data = []
    
    for file in glob.glob(f"{data_dir}/*.csv"):
        prov_id = int(os.path.basename(file).split('_')[2])
        
        with open(file, 'r') as f:
            for line in f:
                clean_line = line.replace('<tt><pre>', '').replace('</pre></tt>', '').strip()
                
                if not clean_line or 'year' in clean_line.lower():
                    continue
                    
                parts = [x.strip() for x in clean_line.split(',')]
                
                if len(parts) >= 7:
                    try:
                        vhi_val = float(parts[6])
                        if vhi_val != -1.00:
                            data.append({
                                'NOAA_ID': prov_id,
                                'Year': int(parts[0]),
                                'Week': int(parts[1]),
                                'SMN': float(parts[2]),
                                'SMT': float(parts[3]),
                                'VCI': float(parts[4]),
                                'TCI': float(parts[5]),
                                'VHI': vhi_val
                            })
                    except ValueError:
                        continue
                        
    df = pd.DataFrame(data)
    
    noaa_to_ua = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    ua_names = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'АР Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }
    
    df['Province_ID'] = df['NOAA_ID'].map(noaa_to_ua)
    df['Province_Name'] = df['Province_ID'].map(ua_names)
    df = df.drop('NOAA_ID', axis=1)
    
    return df

df = load_and_clean_data()
df.head()

,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_ID,Province_Name
0,1982,1,0.059,258.24,51.11,48.78,49.95,21,Хмельницька
1,1982,2,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
2,1982,3,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
3,1982,4,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
4,1982,5,0.058,266.42,46.87,28.57,37.72,21,Хмельницька


## Завдання 3.1: Вибірка VHI для області за вказаний рік
Реалізувати процедуру для формування вибірки ряду VHI для заданої області за вказаний рік та протестувати її роботу.

In [6]:
def get_vhi_by_year_and_province(dataframe, province_id, year):
    result = dataframe[(dataframe['Province_ID'] == province_id) & (dataframe['Year'] == year)]
    return result[['Week', 'VHI']]

print("Дані VHI для Вінницької області (ID 1) за 2020 рік:")
vhi_2020_vinnytsia = get_vhi_by_year_and_province(df, 1, 2020)
print(vhi_2020_vinnytsia.head())

Дані VHI для Вінницької області (ID 1) за 2020 рік:
       Week    VHI
34716     1  40.92
34717     2  43.19
34718     3  44.74
34719     4  45.29
34720     5  44.80


## Завдання 3.2: Вибірка VHI за діапазон років для вказаних областей
Реалізувати процедуру для формування вибірки ряду VHI за вказаний діапазон років для масиву заданих областей та протестувати її роботу.

In [7]:
def get_vhi_by_range(dataframe, province_ids, year_min, year_max):
    result = dataframe[
        (dataframe['Province_ID'].isin(province_ids)) & 
        (dataframe['Year'] >= year_min) & 
        (dataframe['Year'] <= year_max)
    ]
    return result[['Year', 'Week', 'Province_Name', 'VHI']]

print("Дані VHI для Київської (9) та Львівської (12) областей за 2018-2020 роки:")
vhi_range_data = get_vhi_by_range(df, [9, 12], 2018, 2020)
print(vhi_range_data.head())

Дані VHI для Київської (9) та Львівської (12) областей за 2018-2020 роки:
      Year  Week Province_Name    VHI
4008  2018     1      Київська  40.82
4009  2018     2      Київська  43.65
4010  2018     3      Київська  48.62
4011  2018     4      Київська  51.77
4012  2018     5      Київська  52.08


## Завдання 3.3: Пошук екстремумів
Реалізувати процедуру для пошуку екстремумів (min та max VHI) для заданих областей за вказаний діапазон років та вивести результат.

In [8]:
def get_extremes(dataframe, province_ids, year_min, year_max):
    subset = get_vhi_by_range(dataframe, province_ids, year_min, year_max)
    
    stats = {
        'Min VHI': subset['VHI'].min(),
        'Max VHI': subset['VHI'].max(),
        'Mean VHI': subset['VHI'].mean(),
        'Median VHI': subset['VHI'].median()
    }
    return stats

print("Екстремуми для Київської (9) та Львівської (12) областей (2018-2020):")
extremes_data = get_extremes(df, [9, 12], 2018, 2020)
for key, value in extremes_data.items():
    print(f"{key}: {value:.2f}")

Екстремуми для Київської (9) та Львівської (12) областей (2018-2020):
Min VHI: 20.91
Max VHI: 63.62
Mean VHI: 46.78
Median VHI: 46.84
